# 02 — Tiền xử lý dữ liệu 


### 0. Cấu hình và đọc file


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Tự tìm thư mục dự án để notebook chạy ổn định từ mọi thư mục.
CANDIDATES = [root / 'data' / 'raw' / 'RT_IOT2022.csv' for root in [Path.cwd(), *Path.cwd().parents]]
INPUT_PATH = next((p for p in CANDIDATES if p.is_file()), None)
if INPUT_PATH is None:
    raise FileNotFoundError('Không tìm thấy data/raw/RT_IOT2022.csv.')
PROJECT_ROOT = next(p.parent.parent.parent for p in CANDIDATES if p == INPUT_PATH)
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'experiments' / 'baseline' / 'processed' / 'RT_IOT2022_pre_split_12_classes.csv'
REPORT_PATH = PROJECT_ROOT / 'reports' / 'baseline' / 'metrics' / 'preprocessing_audit.csv'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
TARGET = 'Attack_type'
df = pd.read_csv(INPUT_PATH, low_memory=False)
assert TARGET in df.columns
print('Đầu vào:', INPUT_PATH, df.shape)
display(df.head())

## 1. Kiểm tra đầu vào và 12 nhãn gốc (EDA → kiểm chứng)

In [ ]:
expected_labels = set(df[TARGET].dropna().unique())
assert len(expected_labels) == 12, f'Phát hiện {len(expected_labels)} nhãn, cần kiểm tra lại.'
assert df[TARGET].notna().all()
print('Missing:', int(df.isna().sum().sum()))
print('Dòng trùng toàn bộ (có index):', int(df.duplicated().sum()))
print('Cột hằng:', df.columns[df.nunique(dropna=False) <= 1].tolist())
display(df[TARGET].value_counts().rename('before'))

## 2. Loại cột index xuất CSV (EDA: cột `Unnamed: 0` không phải đặc trưng mạng)

In [ ]:
working = df.drop(columns=['Unnamed: 0'], errors='ignore').copy()
print('Số cột sau bỏ index:', working.shape[1])

## 3. Xử lý vector đặc trưng trùng nhưng có nhãn mâu thuẫn

Các bản ghi có **toàn bộ đặc trưng giống nhau nhưng nhãn khác nhau** được đánh dấu mơ hồ và loại **toàn bộ nhóm mâu thuẫn**, thay vì tùy ý giữ một nhãn. Đây là quy tắc cố định, không dựa trên phân bố validation/test. Ghi số mẫu theo lớp để kiểm toán.

In [ ]:
feature_cols = [c for c in working.columns if c != TARGET]
conflicting = working.groupby(feature_cols, dropna=False, sort=False)[TARGET].transform('nunique').gt(1)
conflict_by_label = working.loc[conflicting, TARGET].value_counts()
print('Số dòng có vector đặc trưng mâu thuẫn:', int(conflicting.sum()))
display(conflict_by_label.rename('removed_conflict'))
working = working.loc[~conflicting].copy()

## 4. Loại dòng trùng sau khi bỏ index (EDA: trùng nội dung)

In [ ]:
before_duplicates = len(working)
working = working.drop_duplicates().copy()
removed_duplicates = before_duplicates - len(working)
print('Số dòng trùng đã loại:', removed_duplicates)

## 5. Loại cột hằng (EDA: không có biến thiên)

Chỉ loại cột có đúng một giá trị duy nhất, không sử dụng nhãn để chọn đặc trưng.

In [ ]:
constant_cols = [c for c in working.columns if c != TARGET and working[c].nunique(dropna=False) <= 1]
print('Cột hằng:', constant_cols)
working = working.drop(columns=constant_cols)

## 6. Chuẩn hóa danh mục bằng quy tắc cố định (EDA: `service = "-"`)

`-` là ký hiệu không xác định; đổi thành `unknown` để dễ diễn giải, **không** tự suy luận dịch vụ. Không thay đổi nhãn `Attack_type`.

In [ ]:
for col in ['proto','service']:
    if col in working:
        working[col] = working[col].astype('string').str.strip()
service_dash = int(working['service'].eq('-').sum())
working['service'] = working['service'].replace({'-':'unknown'})
print('service "-" được đổi:', service_dash)

## 7. Kiểm tra đầu ra và báo cáo truy vết EDA → tiền xử lý

In [ ]:
processed = working.reset_index(drop=True)
assert set(processed[TARGET].unique()) == expected_labels
assert processed[TARGET].nunique() == 12
assert not processed.duplicated().any()
assert processed.isna().sum().sum() == 0, 'Có missing values; cần đánh giá nguyên nhân thay vì tự điền toàn cục.'
numeric = processed.select_dtypes(include='number')
assert not np.isinf(numeric.to_numpy()).any(), 'Có Inf; cần xử lý theo quy tắc xác định.'
assert not processed['service'].eq('-').any()
assert not any(c.startswith('Unnamed:') for c in processed.columns)

summary = pd.DataFrame([
    ['EDA: index thừa', 'Bỏ Unnamed: 0', int('Unnamed: 0' in df.columns)],
    ['EDA: nhãn mâu thuẫn cùng vector', 'Loại toàn bộ dòng thuộc nhóm mâu thuẫn', int(conflicting.sum())],
    ['EDA: dòng trùng', 'Bỏ trùng sau khi bỏ index', int(removed_duplicates)],
    ['EDA: cột hằng', 'Bỏ cột hằng', len(constant_cols)],
    ['EDA: service = -', 'Đổi sang unknown', service_dash],
    ['EDA: mất cân bằng', 'Giữ nguyên; chỉ xử lý trên train', 0],
    ['EDA: outlier', 'Giữ nguyên; không loại tự động', 0],
    ['EDA: tương quan', 'Chỉ phân tích; chưa chọn đặc trưng', 0],
], columns=['eda_finding','preprocessing_decision','affected_count'])
display(summary)
display(pd.concat([df[TARGET].value_counts().rename('before'), processed[TARGET].value_counts().rename('after')],axis=1).fillna(0).astype(int))
print('Kích thước:', df.shape, '→', processed.shape)

## 8. Xuất CSV + kiểm chứng đọc lại

In [ ]:
processed.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
summary.to_csv(REPORT_PATH, index=False, encoding='utf-8-sig')
reloaded = pd.read_csv(OUTPUT_PATH, low_memory=False)
assert reloaded.shape == processed.shape
assert reloaded[TARGET].value_counts().to_dict() == processed[TARGET].value_counts().to_dict()
assert reloaded.columns.tolist() == processed.columns.tolist()
print('Đã xuất:', OUTPUT_PATH.resolve())
print('Báo cáo quyết định:', REPORT_PATH.resolve())